# Imports
Imports required to run the notebook

In [1]:
import pandas as pd
import sys
sys.path.append('..')
from pipeline.preprocessors.revised_missing import RevisedMissingValueHandler
from pipeline.preprocessors.numerical import Scaler
from pipeline.preprocessors.categorical import CategoricalEncoder
from pipeline.preprocessors.features import FeatureEngineer
from pipeline.pipeline import PreprocessingPipeline

# Load Dataset
Load the diabetes dataset

In [2]:
df = pd.read_csv('../../data/diabetes.csv')

# Declare 'Impossibles'
Declare the columns that contain values that are biologically impossible to be 0, indicating missing values.

In [3]:
impossibles = ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI']

# Pipeline Implementation
Complete re-usable pipeline implementation using all pipelines created.

In [4]:
pipeline = PreprocessingPipeline(
    missing_value_handler=RevisedMissingValueHandler(
        drop_cols=['Insulin'],
        impute_cols=impossibles,
        zero_cols=impossibles
    ),
    numerical_scaler=Scaler(
        scale_cols=['Glucose', 'BloodPressure', 'SkinThickness',
                    'BMI', 'Pregnancies', 'DiabetesPedigreeFunction', 'Age']
    ),
    categorical_encoder=CategoricalEncoder(
        nominal_cols=[],
        ordinal_cols={'BMICategory': ['Underweight', 'Healthy', 'Overweight', 'Obese']}
    ),
    feature_engineer=FeatureEngineer()
)

df_clean = pipeline.fit_transform(df)
df_clean.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age,Outcome,BMICategory
0,0.635626,0.857470,-0.029161,6.501609e-01,0.163790,0.450630,1.423376,1,3.0
1,-0.848175,-1.202734,-0.515471,-1.706081e-02,-0.855381,-0.375786,-0.194857,0,2.0
2,1.229146,2.002028,-0.677574,3.950746e-16,-1.335847,0.585372,-0.109687,1,1.0
3,-0.848175,-1.071927,-0.515471,-6.842825e-01,-0.636987,-0.926730,-1.046559,0,2.0
4,-1.144936,0.497752,-2.622814,6.501609e-01,1.546951,5.424097,-0.024517,1,3.0


In [5]:
df_clean.shape

(733, 9)

In [6]:
df_clean.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
BMICategory                 0
dtype: int64

# Zero check after scaling
- Zeros in scaled columns represent patients with exactly average values,
not missing data.
- This is expected behaviour from StandardScaler.
Outcome zeros are legitimate (non-diabetic patients).
- BMICategory zeros are legitimate (Underweight = 0 in ordinal encoding).
Pipeline confirmed working correctly.

In [7]:
(df_clean == 0).sum()

Pregnancies                   0
Glucose                       0
BloodPressure                 2
SkinThickness                 0
BMI                           0
DiabetesPedigreeFunction      0
Age                           0
Outcome                     481
BMICategory                   4
dtype: int64

In [8]:
df_clean.describe().round(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age,Outcome,BMICategory
count,733.00,733.00,733.00,733.00,733.00,733.00,733.00,733.00,733.00
mean,-0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,0.34,2.48
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.48,0.74
min,-1.14,-2.54,-3.92,-2.46,-2.08,-1.19,-1.05,0.00,0.00
25%,-0.85,-0.71,-0.68,-0.57,-0.72,-0.69,-0.79,0.00,2.00
50%,-0.25,-0.16,-0.03,0.00,-0.01,-0.29,-0.37,0.00,3.00
75%,0.64,0.63,0.62,0.43,0.60,0.46,0.66,1.00,3.00
max,3.90,2.53,4.02,7.77,5.04,5.82,4.06,1.00,3.00


In [9]:
df_clean['BMICategory'].value_counts()

BMICategory
3.0    459
2.0    173
1.0     97
0.0      4
Name: count, dtype: int64

In [12]:
print(df_clean[df_clean['BloodPressure'] == 0.0])

     Pregnancies   Glucose  BloodPressure  SkinThickness       BMI  \
172    -0.551415 -1.137331            0.0      -0.684282 -0.520510   
357     2.712947  0.236139            0.0       0.094143  1.081044   

     DiabetesPedigreeFunction       Age  Outcome  BMICategory  
172                  0.887792 -0.705878        0          2.0  
357                  0.276963  0.912355        1          3.0  
